# Ciência da Ciência — Cibernética · Regulação · Política Industrial
**IPEA / DIEST-COGIT** · Notebook replicável e inspecionável · v2

---

## Como usar

Execute as células em ordem. Cada fase salva um **ponto de verificação** em parquet — você pode retomar de qualquer ponto sem reexecutar tudo.

| Célula | Fase | Ponto de verificação | Tempo |
|--------|------|------------|-------|
| 1 | Instalação + config | — | 1 min |
| 2 | Obras-semente + vocabulários + utilitários | — | — |
| 3 | **TESTE DE SANIDADE** | — | 1 min |
| 4 | Consultas de texto (fase 1) | `ckpt_phase1.parquet` | 5 min |
| 5 | Bola de neve progressiva (fase 2) | `ckpt_phase2.parquet` | 10 min |
| 6 | Bola de neve regressiva (fase 3) | `ckpt_phase3.parquet` | 8 min |
| 7 | Enriquecimento de refs | `ckpt_enriched.parquet` | 10 min |
| 8 | Cocitação + Acoplamento + Leiden + Intermediação | `ckpt_analysis.parquet` | 5 min |
| 9 | Detecção de rajadas (Kleinberg) + Coeficiente de beleza | `ckpt_metrics.parquet` | 15 min |
| 9b | Rede de cocitação real → `data/network.json` | `network.json` | 1 min |
| 10 | Inspeção interativa dos resultados | — | — |
| 11 | Relatório — **aposentado**: o site é o relatório (ver 13) | — | — |
| 11b | **Exportação completa** (CSVs + ZIP) | ZIP | 1 min |
| 12 | Baixar / salvar no Drive | — | — |
| 13 | **Gera o site camera-ready** em `docs/` (`build_site.py`) | `docs/` | 1 min |

> **Gerar o site camera-ready:** os resultados desta execução (`data/scisci_results.json` + `data/network.json`) alimentam o site em `docs/`. A **Célula 13** o regenera com `python src/build_site.py` (só biblioteca padrão). O caminho mais simples é **clonar o repositório** — que já traz `src/`, `docs/vendor/` e os dados auxiliares — e rodar a Célula 13; daí sai o site completo: relatório, explorador, triagem e exportações Rayyan.

## Referências metodológicas

- **Bola de neve**: Wohlin (2014) *Guidelines for snowballing in systematic literature studies*
- **Força de associação**: Van Eck & Waltman (2009) *How to normalize cooccurrence data* · JASIST
- **Agrupamento de Leiden**: Traag, Waltman & van Eck (2019) *From Louvain to Leiden* · Scientific Reports
- **Detecção de rajadas**: Kleinberg (2003) *Bursty and Hierarchical Structure in Streams* · DMKD
- **Coeficiente de beleza**: Ke et al. (2015) *Defining and identifying sleeping beauties in science* · PNAS 112:7426–7431


## Célula 1 — Instalação e configuração

In [ ]:
# Instalar dependências (versões fixadas para reprodutibilidade)
!pip install -q \
    "python-igraph==0.11.6" \
    "leidenalg==0.10.2" \
    "ipysigma==0.24.6" \
    "pyvis==0.3.2" \
    "plotly>=5.20" \
    "scipy>=1.11" \
    "scikit-learn>=1.3" \
    "pandas>=2.1" \
    "numpy>=1.24" \
    "pyarrow>=14" \
    "requests>=2.31" \
    "tqdm"

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import requests, time, json, warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np, pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix, diags as sp_diags
from scipy.special import gammaln
import igraph as ig, leidenalg as la
from sklearn.feature_extraction.text import TfidfVectorizer
import plotly.express as px, plotly.graph_objects as go

warnings.filterwarnings('ignore')

OUT  = Path("scisci_output"); OUT.mkdir(exist_ok=True)
CKPT = OUT

# ── Parâmetros — ajuste aqui antes de rodar ──────────────────────────
EMAIL = "lucasfreire@gmail.com"   # seu email para polite pool OpenAlex
BASE  = "https://api.openalex.org"
HEADERS = {"User-Agent": f"IPEA-DIEST-COGIT/2.0 (mailto:{EMAIL})"}
SLEEP   = 0.12  # segundos entre requests (polite pool = 100k/dia com email)

P = dict(
    per_page          = 50,    # results por página
    text_pages        = 3,     # páginas por query  → 150 results
    forward_pages     = 6,     # páginas forward    → 300 citantes/seed
    backward_cap      = 400,   # máx refs novas no backward
    backward_anchors  = 60,    # âncoras backward
    cocit_topn        = 600,   # top-N refs para co-citation
    cocit_threshold   = 65,    # percentil de poda co-citation
    bc_min_strength   = 1e-4,  # poda coupling
    bc_threshold_pct  = 60,    # percentil adicional coupling→Leiden
    leiden_resolution = 0.015, # CPM resolution (menor = clusters maiores)
    leiden_seed       = 42,    # reprodutibilidade
    burst_s           = 2.0,   # fator Kleinberg (2 = padrão CiteSpace)
    burst_gamma       = 1.0,   # peso transições Kleinberg
    beauty_min_age    = 8,     # anos mínimos para calcular B
)

print(f"✓ Configuração carregada")
print(f"  Email: {EMAIL}")
print(f"  Colab: {IN_COLAB}")
print(f"  Output: {OUT.resolve()}")
print(f"  Parâmetros chave:")
for k, v in P.items():
    print(f"    {k:<22} = {v}")


## Célula 2 — Obras-semente, vocabulários e utilitários

In [ ]:
# Seeds (IDs OpenAlex verificados em maio 2026)
SEEDS = {
    "Beer_Brain_1972":      ("W2048086870", "Beer, S. (1972). Brain of the Firm"),
    "Beer_Heart_1979":      ("W1566478880", "Beer, S. (1979). The Heart of Enterprise"),
    "Beer_Diagnosing_1985": ("W2154683088", "Beer, S. (1985). Diagnosing the System"),
    "Ashby_1956":           ("W2325487953", "Ashby, W.R. (1956). Introduction to Cybernetics"),
    "Espejo_Walker_2011":   ("W4244612406", "Espejo & Reyes (2011). Organizational Systems"),
    "Hood_1983":            ("W1601629960", "Hood, C. (1983). The Tools of Government"),
    "Hood_Margetts_2007":   ("W2126563689", "Hood & Margetts (2007). Tools Digital Age"),
    "Margetts_2024":        ("W4386803846", "Margetts, H. (2024). Rediscovering Nodality"),
    "Rodrik_2004":          ("W3124879925", "Rodrik, D. (2004). Industrial Policy for the 21C"),
    "Mazzucato_2013":       ("W1553746973", "Mazzucato, M. (2013). The Entrepreneurial State"),
    # Refinamento — Oskar Lange: ponte historica cibernetica x planejamento x socialismo (verificado 2026)
    "Lange_EconCyber_1971": ("W4230710385", "Lange, O.; Nuti, D. M. (1971). Introduction to Economic Cybernetics"),
    "Lange_Socialism_1936": ("W3130930004", "Lange, O. (1936-38). On the Economic Theory of Socialism"),
    "Lange_WholesParts_66": ("W2063282131", "Lange, O. (1966). Wholes and Parts: A General Theory of System Behaviour"),
}
SEED_IDS = [v[0] for v in SEEDS.values()]

# Vocabulários temáticos — expandir para refinar o escopo
VOCAB = {
    "Cyb": {
        "cybernetics","cybernetic","viable system model","viable system","vsm",
        "stafford beer","ashby","requisite variety","algedonic","espejo","espinosa",
        "second-order cybernetic","managerial cybernetic","good regulator theorem",
        "conant-ashby","syntegration","brain of the firm","heart of enterprise",
        "diagnosing the system","viable system approach",
        "economic cybernetics","wholes and parts","general theory of system",
    },
    "Reg": {
        "nodality","tools of government","policy instrument","policy tool",
        "hood and margetts","hood & margetts","policy mix","policy design",
        "regulatory instrument","regulatory capacity","regulatory governance",
        "regulatory state","smart regulation","responsive regulation","meta-regulation",
        "howlett","salamon","instrument choice","governing instrument","policy coherence",
    },
    "PolInd": {
        "industrial policy","developmental state","state capacity",
        "entrepreneurial state","mission-oriented","smart specialisation",
        "smart specialization","neo-developmental","embedded autonomy",
        "industrial strategy","mazzucato","rodrik","dani rodrik",
        "state-led innovation","cybersyn","economic cybernetics",
        "lange cybernetics","optimal planning","market socialism",
        "oskar lange","socialist calculation","lange-lerner","central planning",
    },
}

def axes_of(title, abstract, concepts):
    text = " ".join([str(x or "") for x in [title, abstract, concepts]]).lower()
    return {ax for ax, kws in VOCAB.items() if any(kw in text for kw in kws)}

# Utilitários OpenAlex — com CACHE em disco das consultas (OUT/oa_cache): re-execução
# e depuração não re-consultam a API e sobrevivem a quebras no meio do funil.
import hashlib
OA_CACHE = OUT / "oa_cache"
def _oa_key(url, params):
    raw = url + "?" + "&".join(f"{k}={v}" for k, v in sorted((params or {}).items()))
    return OA_CACHE / (hashlib.sha1(raw.encode()).hexdigest() + ".json")

def oa_get(endpoint, params=None, retries=4, use_cache=True):
    url = endpoint if endpoint.startswith("http") else f"{BASE}/{endpoint}"
    cf = _oa_key(url, params)
    if use_cache and cf.exists():
        try: return json.loads(cf.read_text())
        except Exception: pass
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=30)
            r.raise_for_status(); time.sleep(SLEEP)
            data = r.json()
            if use_cache:
                OA_CACHE.mkdir(exist_ok=True); cf.write_text(json.dumps(data))
            return data
        except requests.HTTPError as e:
            if e.response.status_code == 429:
                wait = 15*(attempt+1); print(f"  429 rate limit, aguardando {wait}s"); time.sleep(wait)
            elif e.response.status_code == 404: return None
            else: raise
        except Exception: time.sleep(3*(attempt+1))
    return None

def oa_paginate(params, max_pages):
    results, p = [], dict(params)
    p.update({"per-page": P["per_page"], "cursor": "*"})
    for _ in range(max_pages):
        data = oa_get("works", p)
        if not data: break
        batch = data.get("results", [])
        if not batch: break
        results.extend(batch)
        nxt = data.get("meta", {}).get("next_cursor")
        if not nxt: break
        p["cursor"] = nxt
    return results

def fetch_batch(ids, batch_size=100):
    out = []
    for i in range(0, len(ids), batch_size):
        chunk = ids[i:i+batch_size]
        if not chunk: continue
        data = oa_get("works", {"filter": f"openalex:{'|'.join(chunk)}", "per-page": batch_size})
        if data: out.extend(data.get("results", []))
    return out

def reconstruct_abstract(inv):
    if not inv or not isinstance(inv, dict): return ""
    n = sum(len(v) for v in inv.values())
    slots = [None]*n
    for word, pos in inv.items():
        for p in pos:
            if 0 <= p < n: slots[p] = word
    return " ".join(w for w in slots if w)

def parse_work(w):
    if w is None: return None
    topics = w.get("topics") or w.get("concepts") or []
    concept_names = [c.get("display_name","") for c in topics[:6]]
    authors = [(a.get("author") or {}).get("display_name","") for a in (w.get("authorships") or [])[:4]]
    authors = [a for a in authors if a]
    venue = ""
    for loc in (w.get("locations") or []):
        src = loc.get("source") or {}
        if src.get("display_name"): venue = src["display_name"]; break
    abstract = reconstruct_abstract(w.get("abstract_inverted_index"))
    refs = [r.split("/")[-1] for r in (w.get("referenced_works") or [])]
    title = w.get("title","") or ""
    ax = axes_of(title, abstract, "; ".join(concept_names))
    return {
        "openalex_id": w.get("id","").split("/")[-1],
        "doi":         w.get("doi","") or "",
        "title":       title,
        "year":        w.get("publication_year"),
        "type":        w.get("type",""),
        "language":    w.get("language",""),
        "cited_by":    int(w.get("cited_by_count") or 0),
        "authors":     "; ".join(authors),
        "venue":       venue,
        "concepts":    "; ".join(concept_names),
        "abstract":    abstract,
        "refs":        refs,
        "n_refs":      len(refs),
        "axes":        ",".join(sorted(ax)),
        "n_axes":      len(ax),
    }

def passes_filter(rec, min_axes=1):
    if rec is None: return False
    year = rec.get("year") or 0
    lang = rec.get("language","")
    typ  = rec.get("type","")
    if year < 1948: return False
    if lang and lang not in ("en","pt","es","fr","de",""): return False
    if typ and typ not in ("article","book","book-chapter","preprint",""): return False
    return rec.get("n_axes",0) >= min_axes

print(f"Seeds: {len(SEEDS)} | Vocab: Cyb={len(VOCAB['Cyb'])} Reg={len(VOCAB['Reg'])} PI={len(VOCAB['PolInd'])} termos")


## Célula 3 — TESTE DE SANIDADE
Execute e verifique que não há erros antes de prosseguir.

In [ ]:
erros = []
print("=" * 60)
print("TESTE DE SANIDADE")
print("=" * 60)

# T1: API respondendo?
t1 = oa_get("works", {"search":"stafford beer","per-page":1})
if t1 and t1.get("meta",{}).get("count",0) > 0:
    print(f"[T1] OK  API respondendo  ({t1['meta']['count']:,} works)")
else:
    erros.append("T1: API falhou"); print("[T1] ERRO: API nao respondeu")

# T2: todas as seeds resolvem?
ids_str = "|".join(SEED_IDS)
t2 = oa_get("works", {"filter": f"openalex:{ids_str}", "per-page": 25})
resolved = {w["id"].split("/")[-1] for w in t2.get("results",[])} if t2 else set()
ok = len(resolved & set(SEED_IDS))
print(f"[T2] {'OK' if ok==len(SEED_IDS) else 'ATENCAO'}  Seeds resolvidas: {ok}/{len(SEED_IDS)}")
for key, (sid, ref) in SEEDS.items():
    mark = "  ok" if sid in resolved else "  FALTA"
    cit = next((w.get("cited_by_count",0) for w in (t2.get("results",[]) if t2 else []) if w.get("id","").endswith(sid)), 0)
    print(f"  {mark}  {sid}  cit={cit:>5}  {ref[:60]}")

# T3: Filtro temático
print("[T3] Filtro tematico:")
POSITIVOS = [
    ("viable system model applied to public governance","Cyb"),
    ("industrial policy state capacity developmental state","PolInd"),
    ("nodality tools of government policy instruments Hood","Reg"),
]
NEGATIVOS = [
    "deep residual learning image recognition convolutional",
    "global cancer statistics GLOBOCAN epidemiology",
    "mediator moderator variable distinction regression",
]
ok3 = True
for text, expected in POSITIVOS:
    ax = axes_of(text,"","")
    ok = expected in ax
    if not ok: ok3=False; erros.append(f"T3 falso negativo: {expected}")
    print(f"  {'ok' if ok else 'ERRO'}  [{','.join(sorted(ax))}]  {text[:50]}")
for text in NEGATIVOS:
    ax = axes_of(text,"","")
    ok = len(ax)==0
    if not ok: erros.append(f"T3 falso positivo: {text[:40]}")
    print(f"  {'ok' if ok else 'FP!'}  bloqueado={ok}  {text[:50]}")
print(f"  Vocabulario: {'OK' if ok3 else 'VERIFICAR'}")

# T4: Forward mini
t4 = oa_get("works", {"filter":"cites:W1601629960","sort":"cited_by_count:desc","per-page":5})
if t4:
    n_pass = sum(1 for w in t4.get("results",[]) if axes_of(w.get("title",""),"",""))
    print(f"[T4] OK  Forward Hood 1983: {t4['meta']['count']:,} citantes; {n_pass}/5 passam filtro")
    for w in t4.get("results",[]):
        ax = axes_of(w.get("title",""),"","")
        print(f"  {'ok' if ax else '--'}  [{','.join(sorted(ax)) or 'nenhum':15s}]  {w.get('title','')[:50]}")
else:
    erros.append("T4: forward falhou")

# T5: igraph + leidenalg
try:
    g = ig.Graph(6,[(0,1),(1,2),(2,3),(3,4),(4,5),(0,5),(0,3),(1,4)])
    g.es["weight"] = [1.0]*8
    part = la.find_partition(g, la.ModularityVertexPartition, weights="weight", seed=42)
    bc   = np.array(g.betweenness()) / max((6-1)*(6-2)/2, 1)
    print(f"[T5] OK  igraph+leidenalg: {len(part)} clusters  betw={bc.round(3).tolist()}")
except Exception as e:
    erros.append(f"T5: {e}"); print(f"[T5] ERRO: {e}")

# T6: Kleinberg
def _kb_test(r, d, s=2., gamma=1.):
    n=len(r)
    if n==0 or d.sum()==0: return []
    ph=r.sum()/max(d.sum(),1.); k=int(np.ceil(1+np.log(max(n,2))/np.log(s)))+1
    ps=np.array([min(ph*(s**i),1-1e-10) for i in range(k)])
    def ll(t,i):
        ri,di=r[t],d[t]
        return 0. if di==0 else gammaln(di+1)-gammaln(ri+1)-gammaln(di-ri+1)+ri*np.log(ps[i])+(di-ri)*np.log(1-ps[i])
    cost=np.full((n,k),np.inf); prev=np.zeros((n,k),dtype=int)
    for j in range(k): cost[0,j]=-ll(0,j)+gamma*j
    for t in range(1,n):
        for j in range(k):
            vals=[cost[t-1,ii]+gamma*max(0,j-ii)-ll(t,j) for ii in range(k)]
            best=int(np.argmin(vals)); cost[t,j]=vals[best]; prev[t,j]=best
    q=np.zeros(n,dtype=int); q[-1]=int(np.argmin(cost[-1]))
    for t in range(n-2,-1,-1): q[t]=prev[t+1,q[t+1]]
    bursts=[]; ib=False; bl=0; bs=0
    for t in range(n):
        if q[t]>=1 and not ib: ib,bl,bs=True,q[t],t
        elif ib and (q[t]<1 or t==n-1):
            be=t if q[t]<1 else t; w=sum(r[bs:be+1])-sum(d[bs:be+1])*ph
            bursts.append((bl,bs,be,max(0.,float(w)))); ib=False
    return bursts
r_t=np.array([1,1,1,5,10,8,3,1,1],dtype=float); d_t=np.array([50.]*9)
bt = _kb_test(r_t, d_t)
print(f"[T6] OK  Kleinberg: {len(bt)} burst(s) detectado(s)  {[(b[0],round(b[3],1)) for b in bt]}")

# T7: Beauty Coefficient
def _beauty_test(c):
    c=np.asarray(c,dtype=float)
    if len(c)<2 or c.sum()==0: return 0.,0
    tm=int(np.argmax(c))
    if tm==0: return 0.,0
    c0,cm=c[0],c[tm]; t=np.arange(tm+1)
    return float(np.sum(((cm-c0)/tm*t+c0-c[:tm+1])/np.maximum(1.,c[:tm+1]))), tm
B, tm = _beauty_test([0]*12+[150])
print(f"[T7] OK  Beauty Coefficient: B={B:.0f}, t_m={tm}  (esperado B>>10, t_m=12)")

# T8: scipy sparse
A = csr_matrix((np.random.rand(80,40)>0.8).astype(np.float32))
C_test = A.T@A; C_test.setdiag(0)
print(f"[T8] OK  scipy.sparse: {C_test.shape}, {C_test.nnz} arestas no mock")

print()
print("=" * 60)
if erros:
    print(f"TESTE DE SANIDADE: {len(erros)} ERRO(S)")
    for e in erros: print(f"  ERRO: {e}")
    print("Corrija antes de prosseguir.")
else:
    print("TESTE DE SANIDADE: PASSOU  - pode prosseguir")
print("=" * 60)


## Célula 4 — Fase 1: consultas de texto
**Ponto de verificação**: `ckpt_phase1.parquet` — pule para a célula 5 se já executou.

In [ ]:
CKPT1 = CKPT / "ckpt_phase1.parquet"
if CKPT1.exists():
    print(f"Checkpoint encontrado: {CKPT1}")
    phase1_rows = pd.read_parquet(CKPT1).to_dict("records")
    for r in phase1_rows:
        if not isinstance(r.get("refs"), list): r["refs"] = []
    print(f"  {len(phase1_rows)} papers carregados do checkpoint")
else:
    TEXT_QUERIES = {
        "Q1_cyb_gov":       {"search":"cybernetics regulation governance policy state",
                             "filter":"type:article,publication_year:1950-2026","sort":"cited_by_count:desc"},
        "Q2_VSM_policy":    {"search":'"viable system model" governance OR "public policy" OR "industrial policy"',
                             "filter":"type:article","sort":"cited_by_count:desc"},
        "Q3_NATO_tools":    {"search":"tools of government nodality Hood Margetts policy instruments",
                             "filter":"type:article","sort":"cited_by_count:desc"},
        "Q4_reg_cyb":       {"search":'"regulatory cybernetics" OR "cybernetic governance" OR "adaptive regulation"',
                             "filter":"type:article","sort":"publication_year:desc"},
        "Q5_PI_complexity": {"search":'"industrial policy" "state capacity" OR "systems thinking" complexity',
                             "filter":"type:article,publication_year:2000-2026","sort":"cited_by_count:desc"},
        "Q6_concept_VSM":   {"filter":"concepts.id:C164802487","sort":"cited_by_count:desc"},
        "Q7_concept_PI_reg":{"filter":"concepts.id:C87980944|C2984066494",
                             "search":"governance systems cybernetics","sort":"cited_by_count:desc"},
        "Q8_Lange_cyb":     {"search":"Lange economic cybernetics planning feedback","sort":"cited_by_count:desc"},
        "Q9_Espinosa_VSM":  {"search":"Espinosa viable system sustainability governance","sort":"cited_by_count:desc"},
    }

    seen = {}

    def add_work(rec, source):
        wid = rec["openalex_id"]
        if wid not in seen: seen[wid] = {**rec, "_sources": set()}
        seen[wid]["_sources"].add(source)

    print("Fase 1a: queries de texto")
    print(f"{'Query':<22} {'Bruto':>7} {'Filtrado':>9} {'Acum.':>7}")
    print("-"*50)
    for qname, params in TEXT_QUERIES.items():
        results = oa_paginate(params, max_pages=P["text_pages"])
        kept = 0
        for w in results:
            rec = parse_work(w)
            if rec and passes_filter(rec, 1):
                add_work(rec, qname); kept += 1
        print(f"  {qname:<20} {len(results):>7} {kept:>9} {len(seen):>7}")

    print("\nFase 1b: seeds individuais")
    for key, (wid, ref) in SEEDS.items():
        w = oa_get(f"works/{wid}")
        if w:
            rec = parse_work(w)
            if rec:
                rec["axes"] = "Cyb,Reg,PolInd"; rec["n_axes"] = 3
                add_work(rec, f"SEED:{key}")
    print(f"  {len(SEEDS)} seeds adicionadas")

    phase1_rows = []
    for rec in seen.values():
        r = dict(rec)
        r["sources"] = "|".join(sorted(r.pop("_sources")))
        phase1_rows.append(r)

    pd.DataFrame(phase1_rows).to_parquet(CKPT1, index=False)
    print(f"\nCheckpoint salvo: {CKPT1}  ({len(phase1_rows)} papers)")

df1 = pd.DataFrame(phase1_rows)
print(f"\nFase 1 resumo: total={len(df1)} | n_axes>=2: {(df1.n_axes>=2).sum()} | 3 eixos: {(df1.n_axes==3).sum()}")
print(f"Eixos: Cyb={df1.axes.str.contains('Cyb').sum()} Reg={df1.axes.str.contains('Reg').sum()} PI={df1.axes.str.contains('PolInd').sum()}")


## Célula 5 — Fase 2: bola de neve progressiva
**Ponto de verificação**: `ckpt_phase2.parquet`

In [ ]:
CKPT2 = CKPT / "ckpt_phase2.parquet"
if CKPT2.exists():
    print(f"Checkpoint encontrado: {CKPT2}")
    phase2_rows = pd.read_parquet(CKPT2).to_dict("records")
    for r in phase2_rows:
        if not isinstance(r.get("refs"), list): r["refs"] = []
    print(f"  {len(phase2_rows)} papers carregados")
else:
    existing_ids = {r["openalex_id"] for r in phase1_rows}
    fwd_seen = set(existing_ids)
    fwd_rows = []

    print(f"Forward snowball: {len(SEEDS)} seeds x {P['forward_pages']} paginas")
    print(f"{'Seed':<25} {'Citantes':>9} {'Novos':>8}")
    print("-"*46)

    for key, (wid, ref) in SEEDS.items():
        results = oa_paginate({"filter": f"cites:{wid}", "sort":"cited_by_count:desc"},
                              max_pages=P["forward_pages"])
        kept = []
        for w in results:
            rec = parse_work(w)
            if not rec: continue
            if rec["openalex_id"] in fwd_seen: continue
            if not passes_filter(rec, 1): continue
            rec["sources"] = f"FWD:{key}"
            fwd_seen.add(rec["openalex_id"])
            kept.append(rec)
        fwd_rows.extend(kept)
        print(f"  {key:<23} {len(results):>9} {len(kept):>8}")

    phase2_rows = phase1_rows + fwd_rows
    pd.DataFrame(phase2_rows).to_parquet(CKPT2, index=False)
    print(f"\nCheckpoint salvo: {CKPT2}  ({len(phase2_rows)} papers, +{len(fwd_rows)} novos)")

df2 = pd.DataFrame(phase2_rows)
print(f"\nFase 2: total={len(df2)} | n_axes>=2: {(df2.n_axes>=2).sum()} | 3 eixos: {(df2.n_axes==3).sum()}")


## Célula 6 — Fase 3: bola de neve regressiva
**Ponto de verificação**: `ckpt_phase3.parquet`

In [ ]:
CKPT3 = CKPT / "ckpt_phase3.parquet"
if CKPT3.exists():
    print(f"Checkpoint encontrado: {CKPT3}")
    phase3_rows = pd.read_parquet(CKPT3).to_dict("records")
    for r in phase3_rows:
        if not isinstance(r.get("refs"), list): r["refs"] = []
    print(f"  {len(phase3_rows)} papers carregados")
else:
    existing_ids = {r["openalex_id"] for r in phase2_rows}

    p2_sorted = sorted(phase2_rows, key=lambda r: r.get("cited_by",0), reverse=True)
    anchor_ids = list(dict.fromkeys(
        SEED_IDS + [r["openalex_id"] for r in p2_sorted[:P["backward_anchors"]]]
    ))
    print(f"Backward snowball: {len(anchor_ids)} ancoras")

    anchor_works = fetch_batch(anchor_ids)
    bwd_ids = set()
    for w in anchor_works:
        for ref in (w.get("referenced_works") or []):
            bwd_ids.add(ref.split("/")[-1])
    bwd_new = [oid for oid in bwd_ids if oid and oid not in existing_ids]
    print(f"  Refs novas: {len(bwd_new)}")
    if len(bwd_new) > P["backward_cap"]:
        bwd_new = bwd_new[:P["backward_cap"]]
        print(f"  Limitado a {P['backward_cap']}")

    bwd_works = fetch_batch(bwd_new)
    bwd_rows = []; bwd_seen = set(existing_ids)
    for w in bwd_works:
        rec = parse_work(w)
        if not rec: continue
        if rec["openalex_id"] in bwd_seen: continue
        if not passes_filter(rec, 1): continue
        rec["sources"] = "BWD"
        bwd_seen.add(rec["openalex_id"])
        bwd_rows.append(rec)

    phase3_rows = phase2_rows + bwd_rows
    pd.DataFrame(phase3_rows).to_parquet(CKPT3, index=False)
    print(f"\nCheckpoint salvo: {CKPT3}  ({len(phase3_rows)} papers, +{len(bwd_rows)} backward)")

df3 = pd.DataFrame(phase3_rows)
print(f"\nCorpus bruto: {len(df3)} papers")
print(f"  Com abstract: {(df3.abstract.str.len()>50).mean():.0%}")
print(f"  Com refs:     {(df3.n_refs>0).mean():.0%}")
print(f"  n_axes>=2:    {(df3.n_axes>=2).sum()}")


## Célula 7 — Enriquecimento de `referenced_works`
**Ponto de verificação**: `ckpt_enriched.parquet`

In [ ]:
CKPT_E = CKPT / "ckpt_enriched.parquet"
if CKPT_E.exists():
    print(f"Checkpoint encontrado: {CKPT_E}")
    df = pd.read_parquet(CKPT_E)
    df["refs"] = df["refs"].apply(lambda x: x if isinstance(x, list) else [])
    print(f"  {len(df)} papers | refs: {(df.n_refs>0).mean():.0%}")
else:
    df = pd.DataFrame(phase3_rows)
    df["is_seed"] = df["openalex_id"].isin(SEED_IDS)

    need = df[df.n_refs==0]["openalex_id"].tolist()
    need_sorted = (df[df.openalex_id.isin(need)]
                   .sort_values("cited_by", ascending=False)
                   ["openalex_id"].tolist()[:1500])
    print(f"Enriquecendo {len(need_sorted)} papers sem refs...")
    enriched = fetch_batch(need_sorted)
    ref_map = {w.get("id","").split("/")[-1]: [r.split("/")[-1] for r in (w.get("referenced_works") or [])]
               for w in enriched if w.get("referenced_works")}
    print(f"  Refs recuperadas para {len(ref_map)} papers")

    df["refs"]    = df.apply(lambda row: ref_map.get(row.openalex_id, row.refs) if row.n_refs==0 else row.refs, axis=1)
    df["n_refs"]  = df["refs"].apply(len)
    df["is_seed"] = df["openalex_id"].isin(SEED_IDS)

    df.to_parquet(CKPT_E, index=False)
    print(f"\nCheckpoint salvo: {CKPT_E}  ({len(df)} papers | refs: {(df.n_refs>0).mean():.0%})")

print(f"\nAmostra do corpus (top 10 por citacoes):")
print(df.sort_values("cited_by",ascending=False)[["year","cited_by","n_refs","n_axes","axes","title"]].head(10).to_string())


## Célula 8 — Cocitação · Acoplamento · Leiden · Intermediação
**Ponto de verificação**: `ckpt_analysis.parquet`

In [ ]:
CKPT_A = CKPT / "ckpt_analysis.parquet"

print("=== Co-citation (Small 1973) ===")
cite_counts = Counter(ref for refs in df["refs"] for ref in refs)
keep_refs = [w for w,_ in cite_counts.most_common(P["cocit_topn"])]
ref_idx   = {w:i for i,w in enumerate(keep_refs)}

rows, cols = [], []
for p_idx, refs in enumerate(df["refs"]):
    for r in refs:
        j = ref_idx.get(r)
        if j is not None: rows.append(p_idx); cols.append(j)

A    = csr_matrix(([1]*len(rows),(rows,cols)), shape=(len(df),len(keep_refs)), dtype=np.int32)
C_cc = (A.T@A).astype(np.float32); C_cc.setdiag(0); C_cc.eliminate_zeros()
w_r  = np.asarray(A.sum(0)).ravel().astype(np.float32)
S_cc = sp_diags(1/np.maximum(w_r,1)) @ C_cc @ sp_diags(1/np.maximum(w_r,1))
thr  = np.percentile(S_cc.data, P["cocit_threshold"])
S_cc.data[S_cc.data<thr] = 0; S_cc.eliminate_zeros()
print(f"  {len(keep_refs)} nos | {S_cc.nnz//2:,} arestas (pct threshold={P['cocit_threshold']})")

print("=== Bibliographic coupling (Kessler 1963) ===")
df_bc = df[df.n_refs>=3].reset_index(drop=True)
all_refs2 = sorted({r for refs in df_bc["refs"] for r in refs})
r2i = {r:i for i,r in enumerate(all_refs2)}
rb,cb = [],[]
for p,refs in enumerate(df_bc["refs"]):
    for r in refs:
        j=r2i.get(r)
        if j is not None: rb.append(p); cb.append(j)
B_mat = csr_matrix(([1]*len(rb),(rb,cb)), shape=(len(df_bc),len(all_refs2)), dtype=np.float32)
M_bc  = (B_mat@B_mat.T).astype(np.float32); M_bc.setdiag(0); M_bc.eliminate_zeros()
deg   = np.asarray(B_mat.sum(1)).ravel()
S_bc  = sp_diags(1/np.maximum(deg,1)) @ M_bc @ sp_diags(1/np.maximum(deg,1))
S_bc_t = S_bc.copy()
S_bc_t.data[S_bc_t.data < np.percentile(S_bc_t.data, P["bc_threshold_pct"])] = 0
S_bc_t.eliminate_zeros()
print(f"  {len(df_bc)} nos | {S_bc_t.nnz//2:,} arestas (pct threshold={P['bc_threshold_pct']})")

print("=== Leiden clustering (Traag 2019) ===")
def s2g(S, labels):
    coo=S.tocoo(); mask=coo.row<coo.col
    edges=list(zip(coo.row[mask].tolist(),coo.col[mask].tolist()))
    g=ig.Graph(n=S.shape[0],edges=edges,directed=False)
    g.es["weight"]=coo.data[mask].tolist(); g.vs["name"]=list(labels); return g

g_cc  = s2g(S_cc, keep_refs)
part_cc = la.find_partition(g_cc, la.CPMVertexPartition, weights="weight",
                             resolution_parameter=P["leiden_resolution"],
                             seed=P["leiden_seed"], n_iterations=20)
g_cc.vs["cluster"] = part_cc.membership
print(f"  Co-citation: {len(part_cc)} clusters (CPM, res={P['leiden_resolution']})")

g_bc    = s2g(S_bc_t, df_bc["openalex_id"].tolist())
part_bc = la.find_partition(g_bc, la.ModularityVertexPartition, weights="weight",
                             seed=P["leiden_seed"], n_iterations=20)
g_bc.vs["cluster"] = part_bc.membership
df_bc = df_bc.copy(); df_bc["cluster"] = part_bc.membership
print(f"  Coupling:     {len(part_bc)} clusters (Modularity)")

# Cluster labels via TF-IDF
cluster_labels = {}
for cid in sorted(set(part_bc.membership)):
    texts = df_bc[df_bc.cluster==cid]["title"].fillna("").tolist()
    if len(texts) < 2: cluster_labels[cid] = f"C{cid:02d}"; continue
    try:
        tv = TfidfVectorizer(max_features=5, stop_words="english"); tv.fit(texts)
        cluster_labels[cid] = " - ".join(tv.get_feature_names_out()[:3])
    except: cluster_labels[cid] = f"C{cid:02d}"
df_bc["cluster_label"] = df_bc["cluster"].map(cluster_labels)

print("\nClusters principais:")
csizes = Counter(part_bc.membership)
for cid, lbl in sorted(cluster_labels.items()):
    n = csizes.get(cid, 0)
    if n < 5: continue
    print(f"  C{cid:02d} [{n:>4}] {lbl}")

print("\n=== Betweenness centrality (Brandes 2001) ===")
n_cc    = g_cc.vcount()
bc_raw  = g_cc.betweenness(weights="weight", directed=False)
denom   = max((n_cc-1)*(n_cc-2)/2, 1)
bc_norm = np.array(bc_raw, dtype=float) / denom
g_cc.vs["betweenness"] = bc_norm.tolist()
print(f"  Top-5 betweenness normalizada: {sorted(bc_norm,reverse=True)[:5]}")

print("\n=== Metadados top-200 refs co-citadas ===")
top200_ids   = [w for w,_ in cite_counts.most_common(200)]
top200_works = fetch_batch(top200_ids)
ref_meta = {}
for w in top200_works:
    wid = w.get("id","").split("/")[-1]
    auths = [a.get("author",{}).get("display_name","").split()[-1]
             for a in (w.get("authorships") or [])[:2]
             if a.get("author",{}).get("display_name")]
    ref_meta[wid] = {"title":w.get("title","") or "","year":w.get("publication_year"),"authors":"; ".join(auths)}
print(f"  Metadados: {len(ref_meta)}/200")

cc_nodes = pd.DataFrame({
    "ref_id":      keep_refs,
    "n_citations": [cite_counts[r] for r in keep_refs],
    "betweenness": bc_norm,
    "cluster_cc":  part_cc.membership,
})
cc_nodes["title"]   = cc_nodes.ref_id.map(lambda r: ref_meta.get(r,{}).get("title","")[:70])
cc_nodes["year"]    = cc_nodes.ref_id.map(lambda r: ref_meta.get(r,{}).get("year"))
cc_nodes["authors"] = cc_nodes.ref_id.map(lambda r: ref_meta.get(r,{}).get("authors",""))
cc_nodes["burst_total"] = 0.0; cc_nodes["burst_peak"] = 0.0; cc_nodes["pivotal"] = False

cc_nodes.to_parquet(CKPT_A, index=False)
print(f"\nCheckpoint salvo: {CKPT_A}  ({len(cc_nodes)} nos co-citation)")


## Célula 9 — Detecção de rajadas (Kleinberg) + Coeficiente de beleza
**Ponto de verificação**: `ckpt_metrics.parquet`

In [ ]:
CKPT_M = CKPT / "ckpt_metrics.parquet"

# Kleinberg burst detection - implementacao propria (Viterbi DP)
def kleinberg_burst(r, d, s=None, gamma=None):
    """
    Kleinberg (2003) Bursty and Hierarchical Structure in Streams.
    r[t] = citacoes no intervalo t
    d[t] = total documentos no corpus no intervalo t
    Retorna: [(level, begin_idx, end_idx, weight)]
    """
    if s is None: s = P["burst_s"]
    if gamma is None: gamma = P["burst_gamma"]
    n = len(r)
    if n == 0 or d.sum() == 0: return []
    p_hat = r.sum() / max(d.sum(), 1.)
    k = int(np.ceil(1 + np.log(max(n,2)) / np.log(s))) + 1
    ps = np.array([min(p_hat*(s**i), 1-1e-10) for i in range(k)])

    def ll(t, i):
        ri, di = r[t], d[t]
        if di == 0: return 0.
        return (gammaln(di+1) - gammaln(ri+1) - gammaln(di-ri+1)
                + ri*np.log(ps[i]) + (di-ri)*np.log(1-ps[i]))

    cost = np.full((n,k), np.inf); prev = np.zeros((n,k), dtype=int)
    for j in range(k): cost[0,j] = -ll(0,j) + gamma*j
    for t in range(1,n):
        for j in range(k):
            vals = [cost[t-1,ii] + gamma*max(0,j-ii) - ll(t,j) for ii in range(k)]
            best = int(np.argmin(vals)); cost[t,j] = vals[best]; prev[t,j] = best
    q = np.zeros(n, dtype=int); q[-1] = int(np.argmin(cost[-1]))
    for t in range(n-2,-1,-1): q[t] = prev[t+1, q[t+1]]
    bursts = []; ib = False; bl = 0; bs = 0
    for t in range(n):
        if q[t]>=1 and not ib: ib,bl,bs = True,q[t],t
        elif ib and (q[t]<1 or t==n-1):
            be = t if q[t]<1 else t
            w = sum(r[bs:be+1]) - sum(d[bs:be+1])*p_hat
            bursts.append((bl,bs,be,max(0.,float(w)))); ib=False
    return bursts

yr_total  = df["year"].value_counts().to_dict()
yr_sorted = sorted(yr_total)
print(f"Calculando bursts para {len(keep_refs)} refs (s={P['burst_s']}, gamma={P['burst_gamma']})...")

burst_totals, burst_peaks, all_burst_data = {}, {}, []
for ref_id in keep_refs:
    yr_cnt = df[df["refs"].apply(lambda L: ref_id in L)]["year"].value_counts().to_dict()
    r = np.array([yr_cnt.get(y,0)           for y in yr_sorted], dtype=float)
    d = np.array([max(yr_total.get(y,0),1)  for y in yr_sorted], dtype=float)
    bursts = kleinberg_burst(r, d)
    tw = sum(b[3] for b in bursts); pw = max((b[3] for b in bursts), default=0.)
    burst_totals[ref_id] = tw; burst_peaks[ref_id] = pw
    for lv,b_s,b_e,b_w in bursts:
        all_burst_data.append({
            "ref_id": ref_id,
            "begin":  int(yr_sorted[b_s]) if b_s<len(yr_sorted) else b_s,
            "end":    int(yr_sorted[b_e]) if b_e<len(yr_sorted) else b_e,
            "level":  int(lv), "weight": float(b_w),
        })

cc_nodes["burst_total"] = cc_nodes.ref_id.map(burst_totals).fillna(0)
cc_nodes["burst_peak"]  = cc_nodes.ref_id.map(burst_peaks).fillna(0)
burst_df = pd.DataFrame(all_burst_data)
n_bursting = burst_df.ref_id.nunique() if len(burst_df) else 0
print(f"  {len(burst_df)} bursts em {n_bursting} referencias")

PIVOTAL_THR = 0.06
cc_nodes["pivotal"] = (cc_nodes.betweenness >= PIVOTAL_THR) & (cc_nodes.burst_total > 0)
print(f"  Pivotal points (betw>={PIVOTAL_THR} + burst>0): {cc_nodes.pivotal.sum()}")

# Beauty Coefficient (Ke et al. 2015 PNAS Eq. 2)
def beauty_coefficient(c_by_age):
    """
    B = sum [(baseline - c_t) / max(1, c_t)] para t = 0..t_m
    baseline = reta de c_0 a c_{t_m}
    t_m = indice do pico (awakening year)
    """
    c = np.asarray(c_by_age, dtype=float)
    if len(c) < 2 or c.sum() == 0: return 0., 0
    t_m = int(np.argmax(c))
    if t_m == 0: return 0., 0
    c0, cm = c[0], c[t_m]; t = np.arange(t_m+1)
    line = (cm-c0)/t_m*t + c0
    B = np.sum((line - c[:t_m+1]) / np.maximum(1., c[:t_m+1]))
    return float(B), t_m

current_year = datetime.now().year
beauty_rows = []
for _, paper in df[df.year.notna() & (df.n_axes>=1)].iterrows():
    pub_year = int(paper.year); age = current_year - pub_year
    if age < P["beauty_min_age"]: continue
    oid = paper.openalex_id
    cit_years = df[df["refs"].apply(lambda L: oid in L)]["year"].value_counts().to_dict()
    c_by_age  = [cit_years.get(pub_year+t, 0) for t in range(age+1)]
    B, t_m = beauty_coefficient(c_by_age)
    beauty_rows.append({"openalex_id":oid,"title":str(paper.title)[:80],
                        "year":pub_year,"cited_by":int(paper.cited_by),"B":B,"t_m":t_m,"axes":paper.axes})
beauty_df = pd.DataFrame(beauty_rows).sort_values("B", ascending=False)
print(f"\nBeauty Coefficient: {len(beauty_df)} papers analisados")

cc_nodes.to_parquet(CKPT_M, index=False)
print(f"Checkpoint salvo: {CKPT_M}")

print("\nTop 10 sleeping beauties (maior B):")
print(beauty_df[["year","cited_by","B","t_m","title"]].head(10).to_string(index=False))

print("\nTop 10 bursts (maior peso):")
if len(burst_df):
    top_b = (burst_df.sort_values("weight",ascending=False).drop_duplicates("ref_id").head(10)
             .merge(cc_nodes[["ref_id","title","n_citations"]],on="ref_id",how="left"))
    for _, r in top_b.iterrows():
        print(f"  w={r.weight:>6.1f}  [{r.begin}-{r.end}]  {str(r.title)[:55]}")


## Célula 9b — Rede de cocitação para o site
Exporta um subgrafo legível da rede **real** de cocitação (top-N por intermediação) para `data/network.json`, no esquema consumido por `src/build_site.py`. Substitui, no site, o núcleo de 25 nós reconstruído via OpenAlex pela rede de verdade.

In [ ]:
# Rede de cocitação -> data/network.json (esquema do site). Uso no Colab.
import json, os
try:
    TOPN = 80
    refs = cc_nodes.ref_id.tolist()              # ordem alinhada a g_cc.vs
    top  = cc_nodes.sort_values('betweenness', ascending=False).head(TOPN)
    idset = set(top.ref_id); seedset = set(SEED_IDS)
    def _axis(t):
        ax = axes_of(t or '', '', '')
        for a in ('Cyb','Reg','PolInd'):
            if a in ax: return a
        return ''
    nodes = [{'id': r.ref_id,
              'label': (str(getattr(r,'title','')) or r.ref_id)[:42],
              'axis': _axis(getattr(r,'title','')),
              'cited_by': int(getattr(r,'n_citations',0) or 0),
              'year': (int(r.year) if pd.notna(getattr(r,'year', None)) else None),
              'seed': r.ref_id in seedset}
             for r in top.itertuples()]
    links = []
    for e in g_cc.es:
        a, b = refs[e.source], refs[e.target]
        if a in idset and b in idset and a != b:
            links.append({'source': a, 'target': b, 'tipo': 'cocita',
                          'peso': float(e['weight']) if 'weight' in e.attributes() else 1.0})
    os.makedirs('data', exist_ok=True)
    json.dump({'nodes': nodes, 'links': links},
              open('data/network.json','w',encoding='utf-8'), ensure_ascii=False, indent=1)
    print(f'data/network.json: {len(nodes)} nos, {len(links)} arestas (rede de cocitacao real)')
except Exception as ex:
    print('Exportacao da rede pulada:', ex)


## Célula 10 — Inspeção interativa dos resultados

In [ ]:
# Temporal
for ax in ["Cyb","Reg","PolInd"]:
    df[f"has_{ax}"] = df["axes"].apply(lambda s: 1 if ax in (s or "") else 0)
td = {}
for ax in ["Cyb","Reg","PolInd"]:
    for y,v in df[df.year.between(1960,2025)].groupby("year")[f"has_{ax}"].sum().items():
        y=int(y)
        if y not in td: td[y]={"year":y,"Cyb":0,"Reg":0,"PolInd":0}
        td[y][ax]=int(v)
temporal = sorted(td.values(), key=lambda x: x["year"])

def _rgba(hx, a):  # Plotly: fillcolor não aceita hex com alpha (#rrggbbaa) — usa rgba()
    hx = hx.lstrip("#"); return f"rgba({int(hx[0:2],16)},{int(hx[2:4],16)},{int(hx[4:6],16)},{a})"
fig = go.Figure()
for ax, col in [("Cyb","#4b3fc2"),("Reg","#1a7a5e"),("PolInd","#b85c10")]:
    fig.add_trace(go.Scatter(
        x=[d["year"] for d in temporal], y=[d[ax] for d in temporal],
        name=ax, fill="tozeroy", line=dict(color=col,width=1.5),
        fillcolor=_rgba(col, 0.13),
    ))
fig.update_layout(title="Producao por eixo tematico (1960-2025)",
                  height=320, xaxis_title="Ano", yaxis_title="Papers",
                  legend=dict(orientation="h", y=1.05))
fig.show()

print("Top 20 papers por citacoes (excl. seeds, n_axes>=1):")
top20 = (df[~df.is_seed & (df.n_axes>=1)].nlargest(20,"cited_by")
         [["year","cited_by","n_axes","axes","authors","title"]])
top20["authors"] = top20["authors"].str.split(";").str[0].str[:30]
top20["title"]   = top20["title"].str[:70]
print(top20.to_string(index=False))

print(f"\nTrading zones (n_axes >= 2): {(df.n_axes>=2).sum()} papers")
bridges = df[df.n_axes>=2].sort_values("cited_by",ascending=False)
for _, r in bridges.head(15).iterrows():
    print(f"  [{r.axes:20s}] {r.cited_by:>5} | {r.title[:65]}")

print(f"\nClusters (coupling, Leiden): {len(cluster_labels)}")
csizes = Counter(part_bc.membership)
for cid, lbl in sorted(cluster_labels.items()):
    n = csizes.get(cid,0)
    if n < 5: continue
    top3 = df_bc[df_bc.cluster==cid].sort_values("cited_by",ascending=False).head(3)
    print(f"  C{cid:02d} [{n:>4}] {lbl}")
    for _, r in top3.iterrows():
        print(f"         {r.cited_by:>5} cit -- {r.title[:55]}")

# Rede interativa (apenas no Colab)
if IN_COLAB:
    try:
        import networkx as nx
        from ipysigma import Sigma
        PALETTE = ["#4b3fc2","#1a7a5e","#b85c10","#c23f3f","#185fa5",
                   "#d4537e","#5c4033","#2e7d32","#6a1b9a","#00838f"]
        G = nx.Graph()
        for i in range(g_cc.vcount()):
            name = g_cc.vs[i]["name"]
            meta = ref_meta.get(name, {})
            G.add_node(name,
                label  = meta.get("title","")[:50] or name,
                cluster= int(part_cc.membership[i]),
                betw   = float(bc_norm[i]),
                burst  = float(cc_nodes.loc[cc_nodes.ref_id==name,"burst_total"].sum()),
                size   = int(cite_counts[name]),
            )
        for n, d in G.nodes(data=True):
            d["color"] = PALETTE[d["cluster"] % len(PALETTE)]
        coo = S_cc.tocoo()
        VTH = np.percentile(coo.data, 82)
        for i,j,w in zip(coo.row,coo.col,coo.data):
            if i<j and w>=VTH: G.add_edge(g_cc.vs[i]["name"],g_cc.vs[j]["name"],weight=float(w))
        print(f"\nRede interativa: {G.number_of_nodes()} nos, {G.number_of_edges()} arestas")
        Sigma(G, node_size="size", node_color="color", node_label="label",
              edge_size="weight", start_layout=8, height=700)
    except Exception as e:
        print(f"Rede: {e}")
else:
    print("\n(Rede interativa disponivel apenas no Colab)")


## Célula 11 — Relatório
O relatório canônico é o **site** em `docs/`, gerado a partir de `scisci_results.json` na **Célula 13** (`src/build_site.py`). A geração antiga via `report_builder` foi aposentada.

In [ ]:
# A geração de relatório via report_builder foi aposentada.
# O relatório camera-ready é o site em docs/, montado a partir de
# scisci_results.json — ver a Célula 13 (src/build_site.py).
print('Relatório: ver Célula 13 — gera o site (docs/) via src/build_site.py')

## Célula 12 — Baixar e salvar no Drive

## Célula 11b — Exportação completa (bruto + análises)

> Roda esta célula e baixa **1 ZIP** com todos os CSVs.
> No Colab o download começa automaticamente. Zero cliques extras.

| Arquivo | Conteúdo |
|---------|----------|
| `00_registro_execucao.csv` | Parâmetros e métricas de execução |
| `01_corpus_completo.csv` | Todos os trabalhos com metadados + abstract |
| `01b_corpus_reduzido.csv` | Idem sem abstract (mais leve para Excel) |
| `02_obras_ponte.csv` | Zonas de intercâmbio (≥ 2 eixos) |
| `03_cocitacao_nos.csv` | Nós cocitação: intermediação, rajada, agrupamento, pivotal |
| `04_acoplamento_nos.csv` | Trabalhos do corpus com agrupamento Leiden |
| `05_rajadas_kleinberg.csv` | Todos os eventos de rajada |
| `05b_rajadas_top100.csv` | 100 maiores refs por peso de rajada |
| `06_belas_adormecidas.csv` | Coeficiente de beleza B por paper |
| `07_agrupamentos_resumo.csv` | 3 maiores trabalhos por agrupamento |
| `08_obras_semente.csv` | Obras-semente com IDs OpenAlex |
| `09_cocitacao_arestas.csv` | Arestas da rede (para Gephi / VOSviewer) |


In [ ]:
"""
EXPORTAÇÃO COMPLETA — rode esta célula para gerar todos os CSVs + ZIP.
Um único arquivo ZIP para baixar.
"""
import zipfile, io
from pathlib import Path
from datetime import datetime

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M")
EXPORT_DIR = OUT / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

# ── helpers ────────────────────────────────────────────────────────────
def save(df_out, name, description=""):
    path = EXPORT_DIR / f"{name}.csv"
    df_out.to_csv(path, index=False, encoding="utf-8-sig")
    n, m = df_out.shape
    print(f"  ok  {name+'.csv':<40} {n:>5} linhas × {m:>2} colunas  {description}")
    return path

saved = []

# ══════════════════════════════════════════════════════════════════════
# 1. CORPUS COMPLETO
# ══════════════════════════════════════════════════════════════════════
corpus_cols = [
    "openalex_id","doi","title","authors","year","type","language",
    "cited_by","venue","concepts","axes","n_axes","n_refs",
    "is_seed","abstract",
]
corpus_cols = [c for c in corpus_cols if c in df.columns]
df_corpus = df[corpus_cols].copy()
df_corpus["openalex_url"] = "https://openalex.org/" + df_corpus["openalex_id"]
saved.append(save(df_corpus, "01_corpus_completo",
    "todos os papers coletados com metadados"))

# Versão enxuta (sem abstract)
df_corpus_slim = df_corpus.drop(columns=["abstract"], errors="ignore")
saved.append(save(df_corpus_slim, "01b_corpus_reduzido",
    "corpus sem abstract (mais leve para Excel)"))

# ══════════════════════════════════════════════════════════════════════
# 2. TRADING ZONES (bridges, n_axes >= 2)
# ══════════════════════════════════════════════════════════════════════
df_bridges = (df[df.n_axes >= 2]
              .sort_values("cited_by", ascending=False)
              [["openalex_id","title","authors","year","cited_by","axes","n_axes","venue","abstract"]]
              .copy())
df_bridges["openalex_url"] = "https://openalex.org/" + df_bridges["openalex_id"]
saved.append(save(df_bridges, "02_obras_ponte",
    "papers com >= 2 eixos tematicos (trading zones)"))

# ══════════════════════════════════════════════════════════════════════
# 3. NÓS DA REDE CO-CITATION
# ══════════════════════════════════════════════════════════════════════
df_ccnodes = cc_nodes[[
    "ref_id","title","authors","year","n_citations",
    "betweenness","burst_total","burst_peak","pivotal","cluster_cc",
]].copy()
df_ccnodes["betweenness"] = df_ccnodes["betweenness"].round(6)
df_ccnodes["burst_total"]  = df_ccnodes["burst_total"].round(2)
df_ccnodes["burst_peak"]   = df_ccnodes["burst_peak"].round(2)
df_ccnodes["openalex_url"] = "https://openalex.org/" + df_ccnodes["ref_id"]
saved.append(save(df_ccnodes, "03_cocitacao_nos",
    "nos da rede co-citation: betweenness, burst, cluster, pivotal"))

# ══════════════════════════════════════════════════════════════════════
# 4. NÓS DA REDE COUPLING (papers do corpus + cluster)
# ══════════════════════════════════════════════════════════════════════
coupling_cols = [
    "openalex_id","title","authors","year","cited_by",
    "axes","n_axes","n_refs","cluster","cluster_label","venue",
]
coupling_cols = [c for c in coupling_cols if c in df_bc.columns]
df_coupling = df_bc[coupling_cols].copy()
df_coupling["openalex_url"] = "https://openalex.org/" + df_coupling["openalex_id"]
saved.append(save(df_coupling, "04_acoplamento_nos",
    "papers do corpus com cluster Leiden (bibliographic coupling)"))

# ══════════════════════════════════════════════════════════════════════
# 5. BURSTS (eventos Kleinberg)
# ══════════════════════════════════════════════════════════════════════
if len(burst_df):
    df_bursts_full = (burst_df
        .merge(cc_nodes[["ref_id","title","authors","year","n_citations"]],
               on="ref_id", how="left")
        .sort_values("weight", ascending=False)
    )
    df_bursts_full["weight"] = df_bursts_full["weight"].round(2)
    df_bursts_full["openalex_url"] = "https://openalex.org/" + df_bursts_full["ref_id"]
    saved.append(save(df_bursts_full, "05_rajadas_kleinberg",
        "todos os eventos de burst detectados (Kleinberg 2003)"))

    # Resumo: um burst por referencia (o mais forte)
    df_burst_top = (df_bursts_full.sort_values("weight",ascending=False)
                    .drop_duplicates("ref_id")
                    .head(100))
    saved.append(save(df_burst_top, "05b_rajadas_top100",
        "top 100 referencias por peso de burst (um por ref)"))
else:
    print("  --  bursts: nenhum evento detectado")

# ══════════════════════════════════════════════════════════════════════
# 6. SLEEPING BEAUTIES (Beauty Coefficient)
# ══════════════════════════════════════════════════════════════════════
if len(beauty_df):
    df_beauty = beauty_df.copy()
    df_beauty["B"] = df_beauty["B"].round(2)
    df_beauty["openalex_url"] = "https://openalex.org/" + df_beauty["openalex_id"]
    saved.append(save(df_beauty, "06_belas_adormecidas",
        "Beauty Coefficient B (Ke et al. 2015 PNAS Eq.2), t_m = ano do pico"))
else:
    print("  --  beauty_df vazio")

# ══════════════════════════════════════════════════════════════════════
# 7. RESUMO DOS CLUSTERS
# ══════════════════════════════════════════════════════════════════════
from collections import Counter
csizes = Counter(part_bc.membership)
cluster_rows = []
for cid, lbl in sorted(cluster_labels.items()):
    n = csizes.get(cid, 0)
    top3 = (df_bc[df_bc.cluster==cid]
            .sort_values("cited_by", ascending=False)
            .head(3)[["title","cited_by","year","authors"]])
    for rank, (_, r) in enumerate(top3.iterrows(), 1):
        cluster_rows.append({
            "cluster_id":    cid,
            "cluster_label": lbl,
            "cluster_size":  n,
            "top_rank":      rank,
            "title":         str(r.title)[:100],
            "cited_by":      int(r.cited_by),
            "year":          int(r.year) if pd.notna(r.year) else None,
            "authors":       str(r.authors)[:60],
        })
df_clusters = pd.DataFrame(cluster_rows)
saved.append(save(df_clusters, "07_agrupamentos_resumo",
    "top-3 papers por cluster (bibliographic coupling, Leiden)"))

# ══════════════════════════════════════════════════════════════════════
# 8. SEEDS
# ══════════════════════════════════════════════════════════════════════
seeds_rows = []
for key, (wid, ref) in SEEDS.items():
    row_in_df = df[df.openalex_id == wid]
    axis = ("Cyb" if any(x in ref for x in ["Beer","Ashby","Espejo"])
            else "Reg" if any(x in ref for x in ["Hood","Margetts"])
            else "PolInd")
    cited = int(row_in_df.iloc[0].cited_by) if len(row_in_df) else None
    seeds_rows.append({
        "key": key, "openalex_id": wid, "reference": ref,
        "axis": axis, "cited_by": cited,
        "openalex_url": f"https://openalex.org/{wid}",
    })
df_seeds = pd.DataFrame(seeds_rows)
saved.append(save(df_seeds, "08_obras_semente",
    "seeds do pipeline com IDs OpenAlex e eixos"))

# ══════════════════════════════════════════════════════════════════════
# 9. MÉTRICAS DE REDE (nó-a-nó para VOSviewer / Gephi)
# ══════════════════════════════════════════════════════════════════════
# Formato edges co-citation (para Gephi/VOSviewer)
coo = S_cc.tocoo()
threshold_viz = float(np.percentile(coo.data, 80))
edges_data = []
for i, j, w in zip(coo.row, coo.col, coo.data):
    if i < j and w >= threshold_viz:
        edges_data.append({
            "source": g_cc.vs[i]["name"],
            "target": g_cc.vs[j]["name"],
            "weight": round(float(w), 6),
            "source_title": ref_meta.get(g_cc.vs[i]["name"],{}).get("title","")[:60],
            "target_title": ref_meta.get(g_cc.vs[j]["name"],{}).get("title","")[:60],
        })
df_edges = pd.DataFrame(edges_data)
saved.append(save(df_edges, "09_cocitacao_arestas",
    "arestas da rede co-citation (association strength, top 20% pct)"))

# ══════════════════════════════════════════════════════════════════════
# 10. PIPELINE LOG
# ══════════════════════════════════════════════════════════════════════
log_rows = [
    {"parametro": k, "valor": str(v)} for k, v in P.items()
]
log_rows += [
    {"parametro": "corpus_size",       "valor": str(len(df))},
    {"parametro": "n_seeds",           "valor": str(len(SEEDS))},
    {"parametro": "n_axes_gte1",       "valor": str((df.n_axes>=1).sum())},
    {"parametro": "n_axes_gte2",       "valor": str((df.n_axes>=2).sum())},
    {"parametro": "n_axes_3",          "valor": str((df.n_axes==3).sum())},
    {"parametro": "cocit_nodes",       "valor": str(len(keep_refs))},
    {"parametro": "cocit_edges",       "valor": str(S_cc.nnz//2)},
    {"parametro": "coupling_nodes",    "valor": str(len(df_bc))},
    {"parametro": "coupling_edges",    "valor": str(S_bc_t.nnz//2)},
    {"parametro": "clusters_cc",       "valor": str(len(part_cc))},
    {"parametro": "clusters_bc",       "valor": str(len(part_bc))},
    {"parametro": "burst_events",      "valor": str(len(burst_df) if 'burst_df' in dir() else 0)},
    {"parametro": "bursting_refs",     "valor": str(n_bursting if 'n_bursting' in dir() else 0)},
    {"parametro": "beauty_computed",   "valor": str(len(beauty_df) if 'beauty_df' in dir() else 0)},
    {"parametro": "run_timestamp",     "valor": RUN_TS},
    {"parametro": "email",             "valor": EMAIL},
]
df_log = pd.DataFrame(log_rows)
saved.append(save(df_log, "00_registro_execucao",
    "parametros e metricas de execucao do pipeline"))

# ══════════════════════════════════════════════════════════════════════
# ZIP ÚNICO
# ══════════════════════════════════════════════════════════════════════
zip_path = OUT / f"scisci_exports_{RUN_TS}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in saved:
        zf.write(p, p.name)

zip_size = zip_path.stat().st_size // 1024
print(f"\n{'='*55}")
print(f"ZIP gerado: {zip_path.name}  ({zip_size} KB)")
print(f"Contém {len(saved)} CSVs")
print(f"{'='*55}")

# Download com 1 clique
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
    print("\nDownload iniciado automaticamente.")
else:
    print(f"\nArquivo em: {zip_path.resolve()}")
    print("No Colab: a linha files.download() baixa o ZIP automaticamente.")


In [ ]:
print("Arquivos gerados:")
for p in sorted(OUT.iterdir()):
    if not p.name.startswith("."): print(f"  {p.name:<45} {p.stat().st_size//1024:>6} KB")

if IN_COLAB:
    from google.colab import files
    print("\nDescomente para baixar:")
    print("# files.download(str(OUT / 'ckpt_enriched.parquet'))")
    print("# files.download(str(OUT / 'ckpt_metrics.parquet'))")
    # files.download(str(OUT / "ckpt_enriched.parquet"))
    # files.download(str(OUT / "ckpt_metrics.parquet"))

# Google Drive
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# dest = Path('/content/drive/MyDrive/scisci_ipea'); dest.mkdir(exist_ok=True)
# for p in OUT.iterdir():
#     shutil.copy(p, dest/p.name)
# print(f"Salvo no Drive: {dest}")

print("\n" + "="*55)
print("PIPELINE COMPLETO")
print("="*55)
try:
    print(f"  Corpus:            {len(df):>5} papers")
    print(f"  Co-citation nos:   {g_cc.vcount():>5}")
    print(f"  Coupling nos:      {g_bc.vcount():>5}")
    print(f"  Clusters cc:       {len(part_cc):>5}")
    print(f"  Clusters bc:       {len(part_bc):>5}")
    print(f"  Bursts:            {len(burst_df):>5}")
    print(f"  Sleeping beauties: {len(beauty_df):>5}")
except: pass


## Célula 13 — Gerar o site camera-ready
O relatório canônico é o **site** em `docs/`, construído a partir de `scisci_results.json` (não exige rodar o funil de novo).

In [ ]:
# Gera o site camera-ready (no repositório). Fora dele, apenas instrui.
import os, shutil, subprocess, sys
if os.path.exists('src/build_site.py'):
    for cand in ('scisci_results.json',):
        if os.path.exists(cand) and os.path.abspath(cand) != os.path.abspath('data/scisci_results.json'):
            os.makedirs('data', exist_ok=True); shutil.copy(cand, 'data/scisci_results.json')
    subprocess.run([sys.executable, 'src/build_site.py'], check=True)
    print('Site camera-ready gerado em docs/index.html')
else:
    print('No repositório, rode:  python src/build_site.py')
